In [2]:
import sys
!{sys.executable} -m pip install lightgbm

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\manoj\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix

from lightgbm import LGBMClassifier

In [2]:
DATA_PATH = r"C:\Users\manoj\Downloads\Network Traffic Intrusion Detection System using LightGBM\data\processed\final_scaled_dataset.parquet"

df = pd.read_parquet(DATA_PATH)

print("Loaded:", df.shape)

Loaded: (1650000, 17)


In [3]:
X = df.drop("label", axis=1)
y = df["label"]

print(X.shape, y.shape)

(1650000, 16) (1650000,)


In [4]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
oof_probs = np.zeros((len(X), 2))

In [5]:
model = LGBMClassifier(

    n_estimators=800,          # controlled
    learning_rate=0.05,

    num_leaves=31,             # small model
    max_depth=6,               # prevents overgrowth

    subsample=0.8,
    colsample_bytree=0.8,

    class_weight="balanced",   # handles imbalance

    objective="binary",
    random_state=42,
    n_jobs=-1
)

In [7]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):

    print(f"\n🔹 Fold {fold+1}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="binary_logloss"
    )

    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)

    oof_preds[val_idx] = preds
    oof_probs[val_idx] = probs


🔹 Fold 1
[LightGBM] [Info] Number of positive: 996355, number of negative: 323645
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.253313 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3300
[LightGBM] [Info] Number of data points in the train set: 1320000, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

🔹 Fold 2
[LightGBM] [Info] Number of positive: 996355, number of negative: 323645
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.057301 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you ca

In [8]:
print("Confusion Matrix:")
print(confusion_matrix(y, oof_preds))

print("\nClassification Report:")
print(classification_report(y, oof_preds))

Confusion Matrix:
[[ 401377    3179]
 [  15270 1230174]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.98    404556
           1       1.00      0.99      0.99   1245444

    accuracy                           0.99   1650000
   macro avg       0.98      0.99      0.99   1650000
weighted avg       0.99      0.99      0.99   1650000



In [9]:
MODEL_PATH = r"C:\Users\manoj\Downloads\Network Traffic Intrusion Detection System using LightGBM\models"

os.makedirs(MODEL_PATH, exist_ok=True)

model_file = os.path.join(MODEL_PATH, "stage1_lightgbm.pkl")

joblib.dump(model, model_file)

print("Model saved:", model_file)

Model saved: C:\Users\manoj\Downloads\Network Traffic Intrusion Detection System using LightGBM\models\stage1_lightgbm.pkl


In [10]:
oof_df = pd.DataFrame({
    "stage1_p_normal": oof_probs[:, 0],
    "stage1_p_attack": oof_probs[:, 1]
})

oof_file = os.path.join(MODEL_PATH, "stage1_oof_probs.parquet")

oof_df.to_parquet(oof_file, index=False)

print("OOF saved:", oof_file)

OOF saved: C:\Users\manoj\Downloads\Network Traffic Intrusion Detection System using LightGBM\models\stage1_oof_probs.parquet


In [11]:
import os

size_mb = os.path.getsize(model_file) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

Model size: 2.71 MB
